In [ ]:
p_apply = "false"
p_sources = "partner_raw_data,brz_3rd_crawler_poi_stream"
p_tables = ""


In [ ]:
%run NB_LIB_EXTRACT_RAWDATA


In [ ]:
# Môi trường: Fabric DEV. Mục đích: backfill timestamp L1 và cột audit, không DROP.
# Thứ tự: sau khi đã seed registry mới và đã tạm pipeline. Chạy p_apply=false trước.
# Idempotent: bảng đã đúng schema thì SKIPPED. Mỗi bảng là 1 commit overwrite.
# Không in payload, điện thoại, email.

import json

PL_NAME = "PL_VV_TRANSFORM_BRONZE_TO_SILVER_FL_00"
SRC_SCHEMA = "lh_vv_bronze.dbo"
UTC_RE_SQL = "^[0-9]{4}-[0-9]{2}-[0-9]{2}T[0-9]{2}:[0-9]{2}:[0-9]{2}[.][0-9]{3}Z$"
EPOCH_FN = {"EPOCH_S_TS": "timestamp_seconds", "EPOCH_MS_TS": "timestamp_millis", "EPOCH_US_TS": "timestamp_micros"}


def table_exists(fq: str) -> bool:
    try:
        spark.sql(f"DESCRIBE TABLE {fq}").collect()
        return True
    except Exception:
        return False


def current_schema(fq: str) -> dict:
    out = {}
    for row in spark.sql(f"DESCRIBE TABLE {fq}").collect():
        name = row["col_name"]
        if not name or name.startswith("#"):
            break
        out[name] = row["data_type"]
    return out


def target_names(spec: TableSpec) -> list:
    cols = [c.name for c in sorted(spec.columns, key=lambda c: c.column_order)]
    cols += [n for n, _ in spec.tech_columns]
    return cols


def schema_matches(spec: TableSpec, old: dict) -> bool:
    if "_ingested_at" in old:
        return False
    expected = {c.name: c.data_type for c in spec.columns}
    expected.update(dict(spec.tech_columns))
    for name, dtype in expected.items():
        if old.get(name, "").lower() != dtype.lower():
            return False
    return True


def old_source(name: str, old: dict):
    """Cột nguồn trên schema cũ. created_at bigint không được đọc như audit."""
    if name in ("src_created_at", "src_updated_at"):
        legacy = "created_at" if name == "src_created_at" else "updated_at"
        if name in old and old[name].lower() == "string":
            return name
        if legacy in old and old[legacy].lower() != "string":
            return legacy
        return name if name in old else None
    return name if name in old else None


def time_expr(spec_col: ColumnSpec, old: dict) -> str:
    source = old_source(spec_col.name, old)
    if source is None:
        return "CAST(NULL AS TIMESTAMP)"
    dtype = old[source].lower()
    quoted = f"`{source}`"
    if dtype == "string":
        return f"to_utc_timestamp(CAST({quoted} AS TIMESTAMP), current_timezone())"
    if dtype == "timestamp":
        return quoted
    if dtype in {"bigint", "int", "integer", "long", "smallint"} and spec_col.convert_rule in EPOCH_FN:
        fn = EPOCH_FN[spec_col.convert_rule]
        return f"CASE WHEN {quoted} = 0 THEN NULL ELSE {fn}({quoted}) END"
    raise ConfigError(f"{spec_col.name}: không đổi được kiểu cũ {dtype} với rule {spec_col.convert_rule}")


def audit_expr(name: str, old: dict, clock: str) -> str:
    parts = []
    if name in old and old[name].lower() == "string":
        parts.append(f"to_utc_timestamp(CAST(`{name}` AS TIMESTAMP), current_timezone())")
    elif name in old and old[name].lower() == "timestamp":
        parts.append(f"`{name}`")
    if "_ingested_at" in old:
        parts.append("`_ingested_at`")
    parts.append(f"to_utc_timestamp(CAST('{clock}' AS TIMESTAMP), current_timezone())")
    return "coalesce(" + ", ".join(parts) + ")"


def build_select(spec: TableSpec, old: dict, clock: str) -> str:
    chosen = []
    for col in sorted(spec.columns, key=lambda c: c.column_order):
        if col.is_time:
            expr = time_expr(col, old)
        else:
            if col.name not in old:
                raise ConfigError(f"{spec.fq_name}.{col.name}: thiếu cột không phải timestamp")
            if old[col.name].lower() != col.data_type.lower():
                raise ConfigError(
                    f"{spec.fq_name}.{col.name}: kiểu {old[col.name]} khác đích {col.data_type}")
            expr = f"`{col.name}`"
        chosen.append(f"{expr} AS `{col.name}`")
    for name, dtype in spec.tech_columns:
        if name in {"created_at", "updated_at"}:
            expr = audit_expr(name, old, clock)
        elif name not in old:
            raise ConfigError(f"{spec.fq_name}: thiếu cột kỹ thuật {name}")
        else:
            expr = f"`{name}`"
        chosen.append(f"{expr} AS `{name}`")
    consumed = set(target_names(spec))
    if "src_created_at" in consumed:
        consumed.add("created_at")
    if "src_updated_at" in consumed:
        consumed.add("updated_at")
    consumed.add("_ingested_at")
    extras = [n for n in old if n not in consumed]
    for name in extras:
        chosen.append(f"`{name}`")
    return "SELECT " + ", ".join(chosen) + f" FROM {spec.fq_name}"


def run_parity(spec, old, select_sql) -> dict:
    old_bits = ["count(*) AS n_old"]
    new_bits = ["count(*) AS n_new"]
    zeros = []
    for col in spec.columns:
        if not col.is_time:
            continue
        source = old_source(col.name, old)
        if source is None:
            continue
        dtype = old[source].lower()
        if dtype == "string":
            old_bits.append(
                f"sum(CASE WHEN `{source}` IS NOT NULL AND `{source}` <> '' THEN 1 ELSE 0 END) AS old__{col.name}")
            new_bits.append(f"sum(CASE WHEN `{col.name}` IS NOT NULL THEN 1 ELSE 0 END) AS new__{col.name}")
            continue
        if dtype in {"bigint", "int", "integer", "long", "smallint"}:
            old_bits.append(
                f"sum(CASE WHEN `{source}` IS NOT NULL AND `{source}` <> 0 THEN 1 ELSE 0 END) AS old__{col.name}")
            zeros.append(f"sum(CASE WHEN `{source}` = 0 THEN 1 ELSE 0 END) AS zero__{col.name}")
            new_bits.append(f"sum(CASE WHEN `{col.name}` IS NOT NULL THEN 1 ELSE 0 END) AS new__{col.name}")
        elif dtype == "timestamp":
            old_bits.append(f"sum(CASE WHEN `{source}` IS NOT NULL THEN 1 ELSE 0 END) AS old__{col.name}")
            new_bits.append(f"sum(CASE WHEN `{col.name}` IS NOT NULL THEN 1 ELSE 0 END) AS new__{col.name}")
    old_row = spark.sql("SELECT " + ", ".join(old_bits + zeros) + f" FROM {spec.fq_name}").first().asDict()
    new_row = spark.sql("WITH conv AS (" + select_sql + ") SELECT " + ", ".join(new_bits) + " FROM conv").first().asDict()
    keys = ", ".join(f"`{k}`" for k in spec.key_cols)
    dups = spark.sql(
        "WITH conv AS (" + select_sql + f") SELECT count(*) AS n FROM "
        f"(SELECT {keys} FROM conv GROUP BY {keys} HAVING count(*) > 1)"
    ).first()["n"]
    errors = []
    if old_row["n_old"] != new_row["n_new"]:
        errors.append(f"số dòng {old_row['n_old']} -> {new_row['n_new']}")
    if dups:
        errors.append(f"trùng khoá {dups}")
    for key, value in old_row.items():
        if not key.startswith("old__"):
            continue
        name = key[5:]
        if int(value or 0) != int(new_row.get(f"new__{name}") or 0):
            errors.append(f"{name}: non-null {value} -> {new_row.get(f'new__{name}')}")
    zero_sentinels = {k[6:]: int(v or 0) for k, v in old_row.items() if k.startswith("zero__")}
    return {"errors": errors, "rows": int(old_row["n_old"]), "zero_sentinels": zero_sentinels}


def post_check(spec: TableSpec) -> list:
    fq = spec.fq_name
    errors = []
    schema = current_schema(fq)
    if "_ingested_at" in schema:
        errors.append("còn _ingested_at")
    cols = [c.name for c in spec.columns if c.is_time] + ["created_at", "updated_at"]
    for name in cols:
        if schema.get(name, "").lower() != "timestamp":
            errors.append(f"{name}: kiểu {schema.get(name)} không phải timestamp")
    null_audit = spark.sql(
        f"SELECT count(*) AS n FROM {fq} WHERE created_at IS NULL OR updated_at IS NULL"
    ).first()["n"]
    if null_audit:
        errors.append(f"audit null {null_audit}")
    props = {r["key"]: r["value"] for r in spark.sql(f"SHOW TBLPROPERTIES {fq}").collect()}
    for key in ("delta.autoOptimize.optimizeWrite", "delta.autoOptimize.autoCompact"):
        if str(props.get(key, "")).lower() != "true":
            errors.append(f"thiếu {key}")
    return errors


def convert_table(spec: TableSpec, clock: str, apply: bool) -> dict:
    item = {"trg_tbl": spec.trg_tbl, "action": "", "rows": 0, "zero_sentinels": {}, "checks": [], "error": None,
            "v_before": None, "v_after": None}
    if not table_exists(spec.fq_name):
        item["action"] = "MISSING"
        return item
    old = current_schema(spec.fq_name)
    item["v_before"] = spark.sql(f"DESCRIBE HISTORY {spec.fq_name} LIMIT 1").first()["version"]
    if schema_matches(spec, old):
        item["action"] = "SKIPPED"
        item["checks"] = post_check(spec)
        if item["checks"]:
            item["action"] = "FAILED"
            item["error"] = "; ".join(item["checks"])
        return item
    select_sql = build_select(spec, old, clock)
    checked = run_parity(spec, old, select_sql)
    item["rows"] = checked["rows"]
    item["zero_sentinels"] = checked["zero_sentinels"]
    if checked["errors"]:
        item["action"] = "FAILED"
        item["error"] = "; ".join(checked["errors"])
        return item
    if not apply:
        item["action"] = "PLANNED"
        return item
    frame = spark.sql(select_sql)
    (frame.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
        .option("userMetadata", "L1_TS_AUDIT_BACKFILL").saveAsTable(spec.fq_name))
    spark.sql(
        f"ALTER TABLE {spec.fq_name} SET TBLPROPERTIES ("
        "'delta.autoOptimize.optimizeWrite' = 'true', "
        "'delta.autoOptimize.autoCompact' = 'true')"
    )
    item["v_after"] = spark.sql(f"DESCRIBE HISTORY {spec.fq_name} LIMIT 1").first()["version"]
    item["checks"] = post_check(spec)
    item["action"] = "FAILED" if item["checks"] else "CONVERTED"
    if item["checks"]:
        item["error"] = "; ".join(item["checks"])
    return item


assert_utc_format()
apply = to_bool(p_apply)
clock = utc_ts_text(utc_now())
sources = split_csv(p_sources) or ["partner_raw_data", "brz_3rd_crawler_poi_stream"]
wanted = set(split_csv(p_tables))
results = []
for src_tbl in sources:
    optional = ", ".join(
        c if table_has(T_PIPELINE_CONFIG, c) else f"CAST(NULL AS STRING) AS {c}"
        for c in PC_OPTIONAL_COLUMNS
    )
    pc_rows = spark.sql(f"""
        SELECT trg_schema, trg_tbl, priority, {optional}
        FROM {T_PIPELINE_CONFIG}
        WHERE pl_name = :pl AND src_schema = :ss AND src_tbl = :st
    """, args={"pl": PL_NAME, "ss": SRC_SCHEMA, "st": src_tbl}).collect()
    reg_rows = spark.sql(f"""
        SELECT src_object_schema, src_object, trg_schema, trg_tbl, trg_column, json_path, data_type,
               convert_rule, is_key, key_order, is_toast, column_order, is_active
        FROM {T_REGISTRY}
        WHERE src_schema = :ss AND src_tbl = :st
    """, args={"ss": SRC_SCHEMA, "st": src_tbl}).collect()
    specs, _configured = build_table_specs(
        [r.asDict() for r in pc_rows], [r.asDict() for r in reg_rows], list(wanted), None,
    )
    for spec in specs:
        if wanted and spec.trg_tbl not in wanted:
            continue
        print(f"--- {spec.fq_name}")
        try:
            item = convert_table(spec, clock, apply)
        except Exception as exc:
            item = {"trg_tbl": spec.trg_tbl, "action": "FAILED", "rows": 0, "zero_sentinels": {},
                    "checks": [], "error": short_error(exc), "v_before": None, "v_after": None}
        results.append(item)
        print(item["action"], item.get("error") or "")

status = "SUCCESS" if results and all(r["action"] in {"CONVERTED", "SKIPPED", "MISSING", "PLANNED"} for r in results) else "FAILED"
payload = {"status": status, "apply": apply, "backfill_clock": clock, "tables": results}
print(json.dumps(payload, ensure_ascii=False, default=str))
notebookutils.notebook.exit(json.dumps(payload, ensure_ascii=False, default=str))
